In [ ]:
import os
from pathlib import Path

nb_dir = Path.cwd()
        

target = (nb_dir / '..' / '..').resolve()
os.chdir(target)




In [ ]:
from grasp.graph.graph_storage import GraphStorage, load_graph_storage

gs_path = "data/cadets_e3/graph_storage/cadets_e3_default_experiment_dataset-cadets_e3_context_size-120_step_size-120_graph_storage.pt"
gs: GraphStorage = load_graph_storage(gs_path)


print(gs)
known_executables = gs.train_subject_cmds
print(len(known_executables))

60699


In [ ]:
import time
from urllib.parse import unquote, urlparse

import psycopg2  # type: ignore
from psycopg2 import sql  # type: ignore

from grasp import config
from grasp.schema import DatasetName

user = "postgres"
password = "lolroflomg"
host = config.DB_HOST
port = 9889

base_url = f"postgresql://{user}:{password}@{host}:{port}"
connection_real_data = f"{base_url}/{DatasetName.CADETS_E3.value}"

# Known executable commands from training graph storage
known_executables_set = set(known_executables)
known_executables_list = sorted(known_executables_set)

new_table_name = "subject_node_table"
backup_table_name = f"{new_table_name}_backup"

# Parse connection URL once
parsed = urlparse(connection_real_data)
dbname = parsed.path.lstrip('/') if parsed.path else None
db_user = unquote(parsed.username) if parsed.username else None
db_password = unquote(parsed.password) if parsed.password else None
db_host = parsed.hostname
db_port = parsed.port

t0 = time.perf_counter()
conn = psycopg2.connect(
    dbname=dbname,
    user=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
    application_name="remove_process_unknown_exec_fast",
)

try:
    with conn, conn.cursor() as cur:
        # Fast path: keep one immutable backup, recreate filtered table from it.
        # If backup already exists, reuse it. If not, rename original once.
        cur.execute("SELECT to_regclass(%s)", (new_table_name,))
        has_main = cur.fetchone()[0] is not None # type: ignore
        cur.execute("SELECT to_regclass(%s)", (backup_table_name,))
        has_backup = cur.fetchone()[0] is not None # type: ignore

        if not has_main and not has_backup:
            raise RuntimeError(
                f"Neither '{new_table_name}' nor '{backup_table_name}' exists."
            )

        if has_main and not has_backup:
            t_rename = time.perf_counter()
            cur.execute(
                sql.SQL("ALTER TABLE {} RENAME TO {}").format(
                    sql.Identifier(new_table_name),
                    sql.Identifier(backup_table_name),
                )
            )
            print(
                f"Renamed original table to backup in "
                f"{time.perf_counter() - t_rename:.3f}s"
            )
        elif has_main and has_backup:
            # Keep existing backup as source of truth; refresh working table below.
            print(
                f"Both '{new_table_name}' and '{backup_table_name}' exist; "
                "keeping backup and refreshing working table."
            )

        source_table = backup_table_name if has_backup or has_main else new_table_name

        # Remember node_uuids that will be removed (unknown execs + NULL cmd)
        t_collect = time.perf_counter()
        cur.execute(
            sql.SQL(
                """
                    SELECT hash_id
                    FROM {}
                    WHERE cmd IS NULL OR NOT (cmd = ANY(%s))
                    """
            ).format(sql.Identifier(source_table)),
            (known_executables_list,),
        )
        deleted_node_hash_ids = [row[0] for row in cur.fetchall()]
        print(
            f"Collected {len(deleted_node_hash_ids)} deleted node_hash_ids in "
            f"{time.perf_counter() - t_collect:.3f}s"
        )

        t_rebuild = time.perf_counter()
        cur.execute(
            sql.SQL("DROP TABLE IF EXISTS {}").format(
                sql.Identifier(new_table_name)
            )
        )
        cur.execute(
            sql.SQL("CREATE TABLE {} (LIKE {} INCLUDING ALL)").format(
                sql.Identifier(new_table_name),
                sql.Identifier(source_table),
            )
        )
        cur.execute(
            sql.SQL(
                "INSERT INTO {} SELECT * FROM {} WHERE cmd = ANY(%s)"
            ).format(
                sql.Identifier(new_table_name),
                sql.Identifier(source_table),
            ),
            (known_executables_list,),
        )

        inserted_rows = cur.rowcount
        print(
            f"Rebuilt filtered '{new_table_name}' with {inserted_rows} rows in "
            f"{time.perf_counter() - t_rebuild:.3f}s"
        )

        # Useful sanity numbers
        cur.execute(
            sql.SQL("SELECT COUNT(*) FROM {}").format(
                sql.Identifier(source_table)
            )
        )
        source_count = cur.fetchone()[0] # type: ignore
        print(f"Source rows: {source_count}")
        print(f"Deleted rows: {source_count - inserted_rows}")

finally:
    conn.close()

print(f"Total elapsed: {time.perf_counter() - t0:.3f}s")

Renamed original table to backup in 0.000s
Collected 73 deleted node_hash_ids in 0.028s
Rebuilt filtered 'subject_node_table' with 224073 rows in 1.076s
Source rows: 224146
Deleted rows: 73
Total elapsed: 1.131s


In [4]:
deleted_node_hash_ids

['3b93188bf125a4077048253f5ff1efac676f4633e599892b5a33f43ce44941ae',
 'f33b873c866445a9e0877381ec56e27f73c1915b6ea3b063c6c1c364115c0bdd',
 'b97333579053dd1625bb5f46f0d31a3a127bb73f0402976f2ad7e4a28d9c6c0d',
 '9f6fa511442379205d64d7e27a29e8a5fa1c38bdc1093e87d82ba2040777bcad',
 '2da75cf0302bfadf88563866dc955f98dc557beb5d98f0b470b74f11ffc746b7',
 '73b7ff0b3d9fe5ea64895d9386adceb037719aa74424f9d476b7b5f336deadac',
 '454ac28c84cbf4536008beb2bfc306aee706cdaca219e07fa102c19bac7090a0',
 'e9f39bf1dca4a82b3b7a948d40960febb0977bca3ff353a34172eeda98a2cb3b',
 'f9d84c3231499f977ab892c93fb73ee26f1714adcc0525a0a38cf55df176e7f3',
 'c2ff89083bc224af9190c1af6873c9ebe47c77197980f65b52c36034f8e26036',
 'bacca69339c852dce01b6cef7a9111e62b85744ccece1a59010f5e155df1b466',
 '03a70ae47db6c2df04c46e426c7ff496f7d13fa8c6008347a46b17fdf40fee5a',
 '0257f7a2e62905b6ee1b52316c1518822d177eab27897c9a1b2d365abe98d4ff',
 '5469590e079de40cfd589ba22249159ef934c755a5cdfc19f5ea815bbab5e5a0',
 '9c77597727eec30be90a97ef3462338d

In [ ]:
event_table_name = "event_table"

# Deduplicate once for stable/efficient ANY() checks
deleted_node_hash_ids_list = sorted(set(deleted_node_hash_ids))

t0_event = time.perf_counter()
conn_event = psycopg2.connect(
    dbname=dbname,
    user=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
    application_name="remove_events_with_deleted_nodes_main_only",
)

try:
    with conn_event:  # noqa: SIM117
        with conn_event.cursor() as cur_event:
            # Ensure main table exists
            cur_event.execute("SELECT to_regclass(%s)", (event_table_name,))
            has_event_main = cur_event.fetchone()[0] is not None # type: ignore
            if not has_event_main:
                raise RuntimeError(f"Table '{event_table_name}' does not exist.")

            event_source_table = event_table_name

            # Count source rows
            cur_event.execute(
                sql.SQL("SELECT COUNT(*) FROM {}").format(sql.Identifier(event_table_name))
            )
            event_source_count = cur_event.fetchone()[0] # type: ignore

            # Count rows to delete
            t_event_count = time.perf_counter()
            cur_event.execute(
                sql.SQL(
                    """
                    SELECT COUNT(*)
                    FROM {}
                    WHERE (src_node IS NOT NULL AND src_node = ANY(%s))
                       OR (dst_node IS NOT NULL AND dst_node = ANY(%s))
                    """
                ).format(sql.Identifier(event_table_name)),
                (deleted_node_hash_ids_list, deleted_node_hash_ids_list),
            )
            event_rows_to_delete = cur_event.fetchone()[0] # type: ignore
            print(
                f"Rows to delete from '{event_table_name}': {event_rows_to_delete} "
                f"(counted in {time.perf_counter() - t_event_count:.3f}s)"
            )

            # Delete directly from main table
            t_event_delete = time.perf_counter()
            cur_event.execute(
                sql.SQL(
                    """
                    DELETE FROM {}
                    WHERE (src_node IS NOT NULL AND src_node = ANY(%s))
                       OR (dst_node IS NOT NULL AND dst_node = ANY(%s))
                    """
                ).format(sql.Identifier(event_table_name)),
                (deleted_node_hash_ids_list, deleted_node_hash_ids_list),
            )
            event_deleted_rows = cur_event.rowcount
            event_inserted_rows = event_source_count - event_deleted_rows  # keep existing variable name usable

            print(
                f"Deleted {event_deleted_rows} rows from '{event_table_name}' in "
                f"{time.perf_counter() - t_event_delete:.3f}s"
            )
            print(f"Event source rows: {event_source_count}")
            print(f"Event remaining rows: {event_inserted_rows}")

finally:
    conn_event.close()

print(f"Total event-table elapsed: {time.perf_counter() - t0_event:.3f}s")


Rows to delete from 'event_table': 47952 (counted in 2.306s)
Deleted 47952 rows from 'event_table' in 6.033s
Event source rows: 36484667
Event remaining rows: 36436715
Total event-table elapsed: 11.139s
